# membox — Quickstart

**Production-grade plug-and-play memory for AI agents.**
Zero config, single-file storage, framework-agnostic.

This notebook gets you from `pip install` to a memory-augmented prompt in **5 minutes**.

> Run cells top to bottom. Everything uses an in-memory database (`":memory:"`) so no files are created on disk.


## 0. Install & import

Install from source (the package lives in `src/`):

```bash
uv sync --extra dev
```


In [1]:
# Make the local package importable when running this notebook from the repo root.
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

from membox import Membox, MemoryConfig


## 1. Create a memory

One class, one import. Backed by SQLite (WAL mode). State persists across restarts when you point it at a real file path.


In [2]:
memory = Membox(":memory:")  # use "my_agent.db" to persist to disk
memory

Membox(owner='default', episodes=0, facts=0, db=':memory:')

## 2. Record events (episodic memory)

`record()` stores a timestamped event with an **importance** score (0.0 trivial → 1.0 life-changing) and an optional emotion tag.


In [3]:
memory.record("User got promoted to Director of Engineering!", importance=1.0, emotion="ecstatic")
memory.record("User ordered black coffee", importance=0.3)
memory.record("User's dog Rocky had a vet visit. All clear.", importance=0.5, emotion="relieved")

# Peek at the most recent events
for ep in memory.recent(3):
    print(f"[{ep.emotion or '-':9}] imp={ep.importance:.1f} | {ep.content}")


[relieved ] imp=0.5 | User's dog Rocky had a vet visit. All clear.
[-        ] imp=0.3 | User ordered black coffee
[ecstatic ] imp=1.0 | User got promoted to Director of Engineering!


## 3. Learn facts (semantic memory)

`learn()` stores a `(subject, predicate, object)` triple with automatic **reinforcement** (repeat a fact → confidence rises) and **contradiction** handling (new value → old value deactivated).


In [ ]:
memory.learn("user", "name",      "Pranav",      confidence=0.95)
memory.learn("user", "prefers",   "black coffee", confidence=0.9)

# Reinforcement: same fact again → confidence climbs toward 1.0
fact, action = memory.learn("user", "prefers", "black coffee")
print(f"action={action}, new confidence={fact.confidence:.3f}")

# Contradiction: new value → old fact deactivated, new one becomes active
memory.learn("user", "lives_in", "Delhi", confidence=0.8)
memory.learn("user", "lives_in", "Mumbai")
print("lives_in facts:", [(f.object, f.is_active) for f in memory.about("user") if f.predicate == "lives_in"])

action=reinforced, new confidence=0.915
lives_in facts: [('Mumbai', True)]


## 4. Retrieve memories (smart recall)

`recall()` scores episodes by **Recency × Relevance × Importance** and returns the top-k with a score breakdown.


In [ ]:
results = memory.recall("coffee", k=3)
for r in results:
    print(f"score={r.score:.3f}  R={r.recency:.2f} V={r.relevance:.2f} I={r.importance:.2f}")
    print(f"   → {r.episode.content}")

score=0.790  R=1.00 V=1.00 I=0.30
   → User ordered black coffee
score=0.600  R=1.00 V=0.00 I=1.00
   → User got promoted to Director of Engineering!
score=0.450  R=1.00 V=0.00 I=0.50
   → User's dog Rocky had a vet visit. All clear.


## 5. Build LLM context

`context()` returns a **prompt-ready string** — user profile + relevant memories — that fits inside a token budget. Paste it straight into your system prompt.


In [6]:
context = memory.context("What does the user like?", max_tokens=2000)
print(context)


## User Profile
- user name Pranav (95%)
- user prefers black coffee (92%)
- user lives_in Mumbai (50%)

## Relevant Memories
- (0m ago [ecstatic]) User got promoted to Director of Engineering!
- (0m ago [relieved]) User's dog Rocky had a vet visit. All clear.
- (0m ago) User ordered black coffee


In [7]:
# Plug into any LLM (OpenAI shown; works with Anthropic, Ollama, OpenRouter, ...)
messages = [
    {"role": "system", "content": f"You are a helpful assistant.\n\n{context}"},
    {"role": "user",   "content": "What should I get for dinner?"},
]
messages  # send to: client.chat.completions.create(model=..., messages=messages)


[{'role': 'system',
  'content': "You are a helpful assistant.\n\n## User Profile\n- user name Pranav (95%)\n- user prefers black coffee (92%)\n- user lives_in Mumbai (50%)\n\n## Relevant Memories\n- (0m ago [ecstatic]) User got promoted to Director of Engineering!\n- (0m ago [relieved]) User's dog Rocky had a vet visit. All clear.\n- (0m ago) User ordered black coffee"},
 {'role': 'user', 'content': 'What should I get for dinner?'}]

## 6. Maintenance

Two background jobs keep memory healthy over time:

- `consolidate()` — compress old episodes into durable facts
- `forget()` — prune/archive stale memories (importance-weighted; critical ones survive)


In [8]:
print("consolidate:", memory.consolidate())
print("forget:      ", memory.forget())
print("stats:       ", memory.stats()["episodes"])


consolidate: {'episodes_processed': 0, 'facts_extracted': 0, 'facts': []}
forget:       {'deleted': 0, 'archived': 0, 'kept': 3, 'total_evaluated': 3, 'actions': [ForgetAction(episode_id='c25db59eb9b04d26', action='keep', retention_score=0.7399999704600015, reason='above all thresholds (imp=1.0, age=0d)'), ForgetAction(episode_id='83eb248eaa2045d9', action='keep', retention_score=0.4599999708266681, reason='above all thresholds (imp=0.3, age=0d)'), ForgetAction(episode_id='201afffeabd5462f', action='keep', retention_score=0.5399999710183347, reason='above all thresholds (imp=0.5, age=0d)')]}
stats:        {'total': 3, 'avg_importance': 0.6, 'consolidated': 0, 'unconsolidated': 3, 'earliest': '2026-06-21T21:04:52.561416', 'latest': '2026-06-21T21:04:52.561751'}


## ✅ You're done

You now know the full loop:

```
record → learn → recall → context → [LLM] → record → ... → consolidate / forget
```

**Next:**
- `notebooks/walkthrough.ipynb` — a detailed simple→advanced tour (procedural memory, threads, reflection, temporal facts, embeddings, multi-user, LLM integration).
- `README.md` — full API reference & design notes.
- `demos/demo_openrouter.py` — a live end-to-end demo with a real LLM.
